# ML Algorithm Training Practice

Hands-on practice notebook for interview prep — train and evaluate the four workhorse algorithms most likely to come up: **Linear Regression**, **Logistic Regression**, **K-Means**, and **XGBoost**.

Pairs with [`ML_Algorithms_Cheatsheet.md`](ML_Algorithms_Cheatsheet.md) in this folder for the theory/math side — this notebook is the "run it and see" side.

Run cells top to bottom. If `xgboost` isn't installed, run `!pip install xgboost` in the cell provided below.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing, load_breast_cancer, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve,
    silhouette_score,
)

np.random.seed(42)
%matplotlib inline

## 1. Linear Regression

Predicting a continuous target (median house value) from numeric features. California housing dataset — clean, no missing values, good for a first pass without preprocessing overhead.

In [ ]:
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target
print(X.shape, y.shape)
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

coef_table = pd.Series(lin_reg.coef_, index=X.columns).sort_values(key=abs, ascending=False)
print('Intercept:', lin_reg.intercept_)
coef_table

### Evaluate

Check: R² (variance explained), RMSE (error in the target's own units), and a residual plot — residuals should look randomly scattered around 0 with no funnel shape (a funnel shape means heteroscedasticity, an assumption violation worth naming explicitly in an interview — see [Example_Company/09_Classical_ML_Statistics.md](../Example_Company/09_Classical_ML_Statistics.md)).

In [ ]:
y_pred = lin_reg.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)
print(f'RMSE: {rmse:.3f}')
print(f'R^2: {r2:.3f}')

residuals = y_test - y_pred
plt.scatter(y_pred, residuals, alpha=0.3)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted')
plt.ylabel('Residual')
plt.title('Residual plot — look for random scatter around 0')
plt.show()

## 2. Logistic Regression

Binary classification — breast cancer dataset (malignant vs. benign). Logistic regression is scale-sensitive (gradient-based optimization assumes comparable feature scales), so standardize first.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
Xc, yc = cancer.data, cancer.target
print(Xc.shape, yc.value_counts().to_dict())

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.2, random_state=42, stratify=yc
)

scaler = StandardScaler()
Xc_train_scaled = scaler.fit_transform(Xc_train)
Xc_test_scaled = scaler.transform(Xc_test)

In [ ]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(Xc_train_scaled, yc_train)

yc_pred = log_reg.predict(Xc_test_scaled)
yc_proba = log_reg.predict_proba(Xc_test_scaled)[:, 1]

print('Accuracy:', accuracy_score(yc_test, yc_pred))
print('Precision:', precision_score(yc_test, yc_pred))
print('Recall:', recall_score(yc_test, yc_pred))
print('F1:', f1_score(yc_test, yc_pred))
print('ROC-AUC:', roc_auc_score(yc_test, yc_proba))
print('Confusion matrix:\n', confusion_matrix(yc_test, yc_pred))

### Threshold isn't fixed at 0.5

`predict()` uses a 0.5 cutoff by default. In a fraud/compliance context (see the `Example_Company/` prep folder), you'd tune this threshold against the actual precision/recall trade-off the business cares about — not accept the default blindly.

In [ ]:
fpr, tpr, thresholds = roc_curve(yc_test, yc_proba)
plt.plot(fpr, tpr, label='Logistic Regression')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

## 3. K-Means

Unsupervised clustering on synthetic blobs. The interview-relevant skill isn't calling `.fit()` — it's justifying *k* and validating the clusters are meaningful, not just numerically distinct.

In [ ]:
X_blobs, y_blobs_true = make_blobs(n_samples=500, centers=4, cluster_std=1.0, random_state=42)
X_blobs_scaled = StandardScaler().fit_transform(X_blobs)

plt.scatter(X_blobs_scaled[:, 0], X_blobs_scaled[:, 1], s=15)
plt.title('Raw data (unlabeled)')
plt.show()

### Choosing k: elbow method + silhouette score

Elbow method: plot inertia (within-cluster sum of squares) vs. k, look for the point of diminishing returns. Silhouette score is more objective — it scores how well-separated clusters are (range -1 to 1, higher is better) and doesn't rely on eyeballing a bend in a curve.

In [ ]:
inertias = []
sil_scores = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_blobs_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_blobs_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(k_range), inertias, marker='o')
axes[0].set_title('Elbow method')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')

axes[1].plot(list(k_range), sil_scores, marker='o', color='orange')
axes[1].set_title('Silhouette score')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Score')
plt.tight_layout()
plt.show()

In [ ]:
best_k = 4  # set from the elbow/silhouette plots above
kmeans_final = KMeans(n_clusters=best_k, n_init=10, random_state=42)
cluster_labels = kmeans_final.fit_predict(X_blobs_scaled)

plt.scatter(X_blobs_scaled[:, 0], X_blobs_scaled[:, 1], c=cluster_labels, cmap='viridis', s=15)
plt.scatter(
    kmeans_final.cluster_centers_[:, 0], kmeans_final.cluster_centers_[:, 1],
    c='red', marker='X', s=150, label='centroids',
)
plt.title(f'K-Means (k={best_k})')
plt.legend()
plt.show()

## 4. XGBoost

Gradient-boosted trees — the default industry choice for tabular classification/regression (fraud scoring, ranking, attribute-quality models). Run `!pip install xgboost` first if it isn't already installed.

In [ ]:
# !pip install xgboost
import xgboost as xgb

Xg_train, Xg_val, yg_train, yg_val = train_test_split(
    Xc_train, yc_train, test_size=0.2, random_state=42, stratify=yc_train
)

xgb_clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    early_stopping_rounds=20,
    random_state=42,
)

xgb_clf.fit(
    Xg_train, yg_train,
    eval_set=[(Xg_val, yg_val)],
    verbose=False,
)
print('Best iteration:', xgb_clf.best_iteration)

In [ ]:
xg_pred = xgb_clf.predict(Xc_test)
xg_proba = xgb_clf.predict_proba(Xc_test)[:, 1]

print('Accuracy:', accuracy_score(yc_test, xg_pred))
print('ROC-AUC:', roc_auc_score(yc_test, xg_proba))
print('Confusion matrix:\n', confusion_matrix(yc_test, xg_pred))

In [ ]:
importances = pd.Series(
    xgb_clf.feature_importances_, index=Xc.columns
).sort_values(ascending=False).head(10)

importances.plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('Top 10 feature importances (XGBoost)')
plt.show()

## Interview talking points to rehearse from this notebook

- Linear regression: which assumption would you check first if RMSE looked fine on train but bad on test?
- Logistic regression: why did we scale features here but not for XGBoost? (Distance/gradient-based linear models are scale-sensitive; tree-based models split on raw thresholds and aren't.)
- K-Means: why silhouette score over inertia alone for choosing k?
- XGBoost: why `early_stopping_rounds` — what failure mode does it prevent?

See [`ML_Algorithms_Cheatsheet.md`](ML_Algorithms_Cheatsheet.md) for the full theory behind each, and [`../Example_Company/09_Classical_ML_Statistics.md`](../Example_Company/09_Classical_ML_Statistics.md) for how these get interrogated in an interview specifically.